# Merge North Region States 2025
## Merge State Parquet Files into a Single Raw Dataset

This notebook belongs to the **interim merge** step of the project data
pipeline. It reads the 7 per-state parquet files produced by the ingestion
template (`01_consolidate_base.ipynb`, one run per state, each
producing `<STATE>_ambulatorial_full_2025.parquet` under
`data/interim/ans_ambulatorial_by_state/<STATE>/`) and merges them into a
**single raw parquet for the entire North region**.

### What this notebook does NOT do

No cleaning, no typing, no filtering, no deduplication. Every column stays
exactly as it came out of the per-state files (string dtype, as documented
in their own provenance tables). This mirrors the same reasoning already
applied within each per-state notebook run: treatment logic is applied
**once**, on the fully merged base, rather than being repeated once per
state and risking divergence between the 7 executions. This notebook's only
job is to merge safely and **validate** that nothing was lost or duplicated in the process: a consistent
schema across states, a traceable state identifier, and row counts that reconcile
against the sum of the source files.

## Phase 0 — Setup

**Upstream:** `01_consolidate_base.ipynb`, run once for each of AC, AM, AP, PA, RO, RR, and TO.

**Required inputs:** the seven canonical files under `data/interim/ans_ambulatorial_by_state/<STATE>/<STATE>_ambulatorial_full_2025.parquet`.

**Output:** `data/interim/north_region_ambulatorial_full_2025.parquet`.

**Downstream:** `04_data_treatment.ipynb`. Paths resolve from either the repository root or `notebooks/`.

In [1]:
import os
from pathlib import Path
import time

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

STATES = ["AC", "AM", "AP", "PA", "RO", "RR", "TO"]
YEAR = "2025"

WORKING_DIR = Path.cwd().resolve()
REPO_ROOT = WORKING_DIR.parent if WORKING_DIR.name == "notebooks" else WORKING_DIR
INTERIM_DIR = REPO_ROOT / "data" / "interim"
INTERIM_BY_STATE_DIR = INTERIM_DIR / "ans_ambulatorial_by_state"

INPUT_PATHS = {
    uf: INTERIM_BY_STATE_DIR / uf / f"{uf}_ambulatorial_full_{YEAR}.parquet"
    for uf in STATES
}
OUTPUT_PATH = INTERIM_DIR / f"north_region_ambulatorial_full_{YEAR}.parquet"

missing_inputs = {uf: path for uf, path in INPUT_PATHS.items() if not path.is_file()}
if missing_inputs:
    details = "\n".join(f"- {uf}: {path}" for uf, path in missing_inputs.items())
    raise FileNotFoundError(
        "Missing state-level inputs. Run 01_consolidate_base.ipynb "
        "for each listed state before running notebook 02. Expected paths:\n" + details
    )

for uf, path in INPUT_PATHS.items():
    print(f"  {uf}: found -- {path}")

  AC: found -- data\interim\ans_ambulatorial_by_state\AC\AC_ambulatorial_full_2025.parquet
  AM: found -- data\interim\ans_ambulatorial_by_state\AM\AM_ambulatorial_full_2025.parquet
  AP: found -- data\interim\ans_ambulatorial_by_state\AP\AP_ambulatorial_full_2025.parquet
  PA: found -- data\interim\ans_ambulatorial_by_state\PA\PA_ambulatorial_full_2025.parquet
  RO: found -- data\interim\ans_ambulatorial_by_state\RO\RO_ambulatorial_full_2025.parquet
  RR: found -- data\interim\ans_ambulatorial_by_state\RR\RR_ambulatorial_full_2025.parquet
  TO: found -- data\interim\ans_ambulatorial_by_state\TO\TO_ambulatorial_full_2025.parquet


## Phase 1 — Schema consistency check

Before merging, we confirm that all 7 states share the exact same set of
columns. If a state diverges (a column renamed, missing, or extra), merging
blindly would either silently drop data or introduce a column full of
nulls for the other 6 states.

In [2]:
schemas = {}
for uf, path in INPUT_PATHS.items():
    schemas[uf] = set(pq.ParquetFile(path).schema_arrow.names)

reference_uf = STATES[0]
reference_schema = schemas[reference_uf]

print(f"Reference schema (from {reference_uf}): {len(reference_schema)} columns\n")

divergences_found = False
for uf in STATES[1:]:
    missing = reference_schema - schemas[uf]
    extra = schemas[uf] - reference_schema
    if missing or extra:
        divergences_found = True
        print(f"[DIVERGENCE] {uf}: missing={sorted(missing)} extra={sorted(extra)}")

if not divergences_found:
    print("No schema divergence across the 7 states -- every state has the exact same columns.")

Reference schema (from AC): 33 columns

No schema divergence across the 7 states -- every state has the exact same columns.


If a divergence is ever found here, it must be resolved explicitly (e.g.
renaming a column, or documenting why a state's file is structurally
different) **before** proceeding to the merge, silently continuing past a
schema mismatch is exactly the kind of thing that produces a corrupted
downstream base without anyone noticing until much later.

In [3]:
assert not divergences_found, "Resolve schema divergences above before merging."

## Phase 2 — Merge with incremental writing

Reading all 7 states into memory at once before writing anything would
require holding the full ~660 MB (uncompressed, larger in memory as
pandas objects) all at once, on top of whatever the concatenation step
itself allocates. Following the same reasoning already used in the
per-state ingestion notebook (see its own memory discussion), we read and
write **one row group at a time, not one state at a time**: a single
state's parquet already spans multiple row groups (`pyarrow.parquet.ParquetWriter`
caps each row group at 1,048,576 rows during ingestion, so populous states
such as PA or AM are split into dozens of row groups automatically), and loading a whole state in a single `pq.read_table()` call would hold up to
~28M rows (PA) in memory at once — enough
to exceed available memory on a 16 GB machine despite the intent to bound
memory per state. Reading and writing by row group instead (`ParquetFile.read_row_group(i)`)
bounds peak memory to at most 1,048,576 rows regardless of how large the
source state file is, and each row group is appended to the merged output
via `pyarrow.parquet.ParquetWriter` and freed before the next one is read.

Each state's file already carries its own `UF` and `MES` columns (added
during per-state ingestion), so the state identifier survives the merge
without needing to be re-added here, we only verify it is present and
correct as we go, accumulated across every row group of a state before
moving to the next state.

In [4]:
row_counts_per_state = {}
writer = None
t0 = time.time()

os.makedirs(INTERIM_DIR, exist_ok=True)

for uf in STATES:
    parquet_file = pq.ParquetFile(INPUT_PATHS[uf])
    state_row_count = 0
    state_ufs = set()

    for row_group_index in range(parquet_file.num_row_groups):
        table = parquet_file.read_row_group(row_group_index)
        state_row_count += table.num_rows
        state_ufs.update(table.column("UF").to_pylist())

        if writer is None:
            writer = pq.ParquetWriter(OUTPUT_PATH, table.schema)
        writer.write_table(table)

        del table

    assert state_ufs == {uf}, f"{uf}: UF column contains unexpected values {state_ufs}"
    row_counts_per_state[uf] = state_row_count

    print(
        f"  {uf}: {state_row_count:,} rows merged across {parquet_file.num_row_groups} "
        f"row groups ({time.time() - t0:.0f}s elapsed)"
    )

writer.close()
print(f"\nMerge complete in {time.time() - t0:.0f}s.")

  AC: 1,208,000 rows merged across 12 row groups (5s elapsed)


  AM: 16,536,319 rows merged across 23 row groups (86s elapsed)


  AP: 2,537,634 rows merged across 12 row groups (99s elapsed)


  PA: 27,919,335 rows merged across 34 row groups (239s elapsed)


  RO: 5,607,473 rows merged across 12 row groups (260s elapsed)


  RR: 1,837,310 rows merged across 12 row groups (267s elapsed)


  TO: 4,267,233 rows merged across 12 row groups (282s elapsed)

Merge complete in 283s.


## Phase 3 — Row-count reconciliation

The single most important check for a merge step: the total row count of
the merged file must equal the exact sum of the 7 source files. Any
mismatch here means rows were silently dropped or duplicated during the
merge, and the output should not be trusted or shared downstream until
that is understood.

In [5]:
merged_file = pq.ParquetFile(OUTPUT_PATH)
total_merged_rows = merged_file.metadata.num_rows
total_source_rows = sum(row_counts_per_state.values())

print("Row counts by state (source files):")
for uf, n in row_counts_per_state.items():
    print(f"  {uf}: {n:,}")

print(f"\nSum of source files: {total_source_rows:,}")
print(f"Rows in merged file: {total_merged_rows:,}")

assert total_merged_rows == total_source_rows, (
    "Row count mismatch between the merged file and the sum of the source files -- "
    "investigate before sharing this output downstream."
)
print("\nOK: merged row count exactly matches the sum of the 7 source files.")

Row counts by state (source files):
  AC: 1,208,000
  AM: 16,536,319
  AP: 2,537,634
  PA: 27,919,335
  RO: 5,607,473
  RR: 1,837,310
  TO: 4,267,233

Sum of source files: 59,913,304
Rows in merged file: 59,913,304

OK: merged row count exactly matches the sum of the 7 source files.


In [6]:
output_size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Merged file: {OUTPUT_PATH}")
print(f"Size: {output_size_mb:.1f} MB")
print(f"Row groups: {merged_file.metadata.num_row_groups}")
print(f"Columns: {len(merged_file.schema_arrow.names)}")

Merged file: data\interim\north_region_ambulatorial_full_2025.parquet
Size: 671.6 MB
Row groups: 117
Columns: 33


## Summary

| Item | Result |
|---|---|
| States merged | AC, AM, AP, PA, RO, RR, TO |
| Schema divergences | none found (see Phase 1) |
| Row-count reconciliation | merged total == sum of the 7 source files (see Phase 3) |
| Output | `data/interim/north_region_ambulatorial_full_2025.parquet` |
| Treatment applied | none — this is still a raw file |

### Next step

`04_data_treatment.ipynb` reads this file together with the independent crosswalk produced by `03_tuss_sboc_crosswalk.ipynb`. Notebook 03 does not consume this output. No cleaning, filtering, or typing happens here; that is deliberately deferred to notebook 04, so it is applied exactly once on the complete North region base rather than up to 7 times.

## References

1. Agência Nacional de Saúde Suplementar (ANS). Dados abertos: TISS Ambulatorial [Internet]. Brasília: ANS; 2026 [cited 2026 Sep 11]. Available from: https://dadosabertos.ans.gov.br/FTP/PDA/TISS/AMBULATORIAL/
2. Apache Software Foundation. Apache Parquet: file format documentation [Internet]. 2026 [cited 2026 Sep 11]. Available from: https://parquet.apache.org/docs/
